# 04 — Time split → impute (train-only)

Pipeline khách quan theo `churn_30d_feature_schema.md` §17:

1. Load feature dataset từ notebook 02
2. Xuất **full dataset** (`churn_feature_dataset_full`) — **không** impute missing
3. **Split theo thời gian** (`snapshot_date`) — train / val / test
4. **Purge gap ≥ 30 ngày** (1 snapshot tháng) giữa các khối để tránh chồng nhãn 30d
5. **Impute** median/mode — **fit chỉ trên train**, transform val & test
6. Xuất train/val/test + bảng thống kê impute

| Split | Ý nghĩa |
|---|---|
| train | Quá khứ — fit imputer + model |
| val | Giữa — tune threshold / hyperparam |
| test | Gần nhất — báo cáo cuối (AUCPR, AUC) |

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

PROJECT_ROOT = None
for _p in [Path.cwd(), Path.cwd().parent, Path.cwd() / "notebooks"]:
    for root in [_p, _p.parent]:
        if (root / "data").exists():
            PROJECT_ROOT = root.resolve()
            break
    if PROJECT_ROOT is not None:
        break
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd().resolve()

DATA_DIR = PROJECT_ROOT / "data"
INPUT_PARQUET = DATA_DIR / "churn_feature_dataset.parquet"
INPUT_CSV = DATA_DIR / "churn_feature_dataset.csv"

OUT_FULL_PARQUET = DATA_DIR / "churn_feature_dataset_full.parquet"
OUT_FULL_CSV = DATA_DIR / "churn_feature_dataset_full.csv"
OUT_TRAIN = DATA_DIR / "churn_train.parquet"
OUT_VAL = DATA_DIR / "churn_val.parquet"
OUT_TEST = DATA_DIR / "churn_test.parquet"
OUT_IMPUTE = DATA_DIR / "impute_stats_train.json"

# Tỷ lệ theo số dòng (sau đó cắt theo tháng gần nhất)
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
# Purge: bỏ 1 tháng snapshot giữa các khối (≥ horizon nhãn 30d)
PURGE_MONTHS = 1

DROP_COLS = [
    "customer_id",
    "snapshot_date",
    "snapshot_month",
    "snapshot_month_ord",
    "churn_reason",
]

CATEGORICAL = [
    "is_free_tier", "is_paid_tier", "auto_renew", "subscription_expired",
    "had_downgrade_30d", "had_downgrade_90d", "had_upgrade_90d",
    "has_any_activity_7d", "has_any_activity_14d", "has_any_activity_30d",
    "is_declining_engagement", "reactivation_flag",
    "has_unresolved_ticket", "avg_csat_score_missing",
    "payments_success_rate_missing", "session_duration_trend_missing",
    "has_marketing_click_30d", "has_completed_order",
    "cancel_request_flag_30d", "had_cancel_attempt_90d",
    "free_and_inactive_14d", "free_and_inactive_21d",
    "paid_weak_engagement", "recent_downgrade_and_quiet", "auto_renew_off_paid",
    "subscription_tier", "account_status_at_t", "account_status", "risk_recency_bucket",
    "gender", "region", "city",
]

print("PROJECT_ROOT", PROJECT_ROOT)
print("INPUT_PARQUET", INPUT_PARQUET.exists(), "|", "INPUT_CSV", INPUT_CSV.exists())

## 1. Load dataset

In [ ]:
if INPUT_PARQUET.exists():
    df = pd.read_parquet(INPUT_PARQUET)
    print("loaded parquet", INPUT_PARQUET)
elif INPUT_CSV.exists():
    df = pd.read_csv(INPUT_CSV)
    print("loaded csv", INPUT_CSV)
else:
    raise FileNotFoundError("Chạy notebook 02 trước để tạo data/churn_feature_dataset.parquet|csv")

df["snapshot_date"] = pd.to_datetime(df["snapshot_date"], errors="coerce")
if "snapshot_month" in df.columns:
    df["snapshot_month"] = pd.to_datetime(df["snapshot_month"], errors="coerce")

# Tương thích schema mới (02) và CSV cũ
if "label_churn_30d" in df.columns:
    TARGET = "label_churn_30d"
elif "churn_30d" in df.columns:
    TARGET = "churn_30d"
else:
    raise KeyError("Không tìm thấy label_churn_30d hoặc churn_30d")

extra_drop = [c for c in df.columns if c.startswith("churn_case") or c in {"churn_60d"}]
DROP_COLS_EFF = list(dict.fromkeys(DROP_COLS + extra_drop))

df = df.sort_values(["snapshot_date", "customer_id"]).reset_index(drop=True)
print("shape", df.shape)
print("TARGET", TARGET, "rate", round(df[TARGET].mean(), 4))
print("months", df["snapshot_date"].nunique(),
      "from", df["snapshot_date"].min().date(), "to", df["snapshot_date"].max().date())
print("null % top 12")
print((df.isna().mean() * 100).sort_values(ascending=False).head(12).round(2).to_string())

# Full feature dataset — giữ nguyên NaN (không impute thống kê)
DATA_DIR.mkdir(parents=True, exist_ok=True)
df.to_parquet(OUT_FULL_PARQUET, index=False)
df.to_csv(OUT_FULL_CSV, index=False)
print("\nsaved FULL (no impute):", OUT_FULL_PARQUET.name, "+", OUT_FULL_CSV.name,
      "rows", len(df), "null_cells", int(df.isna().sum().sum()))

## 2. Time split + purge gap

Cắt theo **số tháng snapshot** (~70% / 15% / 15% tháng usable).
Giữa train↔val và val↔test bỏ `PURGE_MONTHS` tháng (≥ horizon nhãn 30d).

In [ ]:
month_counts = (
    df.groupby("snapshot_date", sort=True)
    .size()
    .rename("n_rows")
    .reset_index()
)
months = month_counts["snapshot_date"].tolist()
n_months = len(months)

min_needed = 2 + 2 * PURGE_MONTHS
if n_months < min_needed:
    raise ValueError(f"Cần ≥ {min_needed} tháng để split + purge, hiện có {n_months}")

# Chia theo SỐ THÁNG (tháng sau đông hơn → % rows test sẽ cao hơn % tháng)
usable_budget = n_months - 2 * PURGE_MONTHS
n_train = max(1, int(round(usable_budget * TRAIN_FRAC)))
n_val = max(1, int(round(usable_budget * VAL_FRAC)))
n_test = usable_budget - n_train - n_val
if n_test < 1:
    n_test = 1
    n_train = max(1, usable_budget - n_val - n_test)

train_end_idx = n_train - 1
val_start_idx = train_end_idx + 1 + PURGE_MONTHS
val_end_idx = val_start_idx + n_val - 1
test_start_idx = val_end_idx + 1 + PURGE_MONTHS

train_months = months[: train_end_idx + 1]
purge1_months = months[train_end_idx + 1 : val_start_idx]
val_months = months[val_start_idx : val_end_idx + 1]
purge2_months = months[val_end_idx + 1 : test_start_idx]
test_months = months[test_start_idx:]

train_raw = df[df["snapshot_date"].isin(train_months)].copy()
val_raw = df[df["snapshot_date"].isin(val_months)].copy()
test_raw = df[df["snapshot_date"].isin(test_months)].copy()
purged = df[df["snapshot_date"].isin(purge1_months + purge2_months)]

def _summary(name, part, months_list):
    return {
        "split": name,
        "n_rows": len(part),
        "pct": round(100 * len(part) / len(df), 2),
        "n_months": len(months_list),
        "from": months_list[0].date() if months_list else None,
        "to": months_list[-1].date() if months_list else None,
        "churn_rate": round(part[TARGET].mean(), 4) if len(part) else None,
    }

split_summary = pd.DataFrame([
    _summary("train", train_raw, train_months),
    _summary("purge_1", purged[purged["snapshot_date"].isin(purge1_months)], purge1_months),
    _summary("val", val_raw, val_months),
    _summary("purge_2", purged[purged["snapshot_date"].isin(purge2_months)], purge2_months),
    _summary("test", test_raw, test_months),
])
print(split_summary.to_string(index=False))
print("\npurged rows (không dùng train/val/test):", len(purged))
print(f"month budget: train={n_train} val={n_val} test={len(test_months)} purge={2 * PURGE_MONTHS} / total={n_months}")

assert train_raw["snapshot_date"].max() < val_raw["snapshot_date"].min()
assert val_raw["snapshot_date"].max() < test_raw["snapshot_date"].min()
print("QA: thứ tự thời gian train < val < test — OK")

## 3. Impute — fit trên train, apply val/test

- **Numeric** còn NaN → median train
- **Categorical / object** còn NaN → mode train (fallback `"missing"`)
- Flag `*_missing` giữ nguyên (không impute lại theo thống kê toàn bộ)
- Không đụng `TARGET` / key meta

In [ ]:
feature_cols = [c for c in df.columns if c not in DROP_COLS_EFF and c != TARGET]
cat_cols = [c for c in feature_cols if c in CATEGORICAL or df[c].dtype == "object" or str(df[c].dtype) == "string"]
num_cols = [c for c in feature_cols if c not in cat_cols]

impute_stats = {"target": TARGET, "numeric_median": {}, "categorical_mode": {}}

# Fit trên train
for col in num_cols:
    if train_raw[col].isna().any():
        med = float(train_raw[col].median())
        if np.isnan(med):
            med = 0.0
        impute_stats["numeric_median"][col] = med

for col in cat_cols:
    if train_raw[col].isna().any():
        mode = train_raw[col].mode(dropna=True)
        fill = mode.iloc[0] if len(mode) else "missing"
        # JSON-serializable
        if hasattr(fill, "item"):
            fill = fill.item()
        impute_stats["categorical_mode"][col] = fill if pd.notna(fill) else "missing"

print("numeric cols to impute:", len(impute_stats["numeric_median"]))
print(pd.Series(impute_stats["numeric_median"]).sort_index().to_string() if impute_stats["numeric_median"] else "(none)")
print("\ncategorical cols to impute:", len(impute_stats["categorical_mode"]))
print(pd.Series(impute_stats["categorical_mode"]).sort_index().to_string() if impute_stats["categorical_mode"] else "(none)")


def apply_impute(part: pd.DataFrame) -> pd.DataFrame:
    out = part.copy()
    for col, med in impute_stats["numeric_median"].items():
        if col in out.columns:
            out[col] = out[col].fillna(med)
    for col, mode in impute_stats["categorical_mode"].items():
        if col in out.columns:
            out[col] = out[col].fillna(mode)
    return out


train = apply_impute(train_raw)
val = apply_impute(val_raw)
test = apply_impute(test_raw)

def null_report(name, before, after):
    b = before[feature_cols].isna().sum().sum()
    a = after[feature_cols].isna().sum().sum()
    print(f"{name}: null cells before={b:,} after={a:,}")

null_report("train", train_raw, train)
null_report("val", val_raw, val)
null_report("test", test_raw, test)

remaining = {
    "train": int(train[feature_cols].isna().sum().sum()),
    "val": int(val[feature_cols].isna().sum().sum()),
    "test": int(test[feature_cols].isna().sum().sum()),
}
assert remaining["train"] == 0, remaining
assert remaining["val"] == 0, remaining
assert remaining["test"] == 0, remaining
print("QA: feature nulls = 0 trên train/val/test — OK")

## 4. Lưu train / val / test + impute stats

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)

meta = {
    "target": TARGET,
    "train_frac": TRAIN_FRAC,
    "val_frac": VAL_FRAC,
    "purge_months": PURGE_MONTHS,
    "train_from": str(train_months[0].date()),
    "train_to": str(train_months[-1].date()),
    "val_from": str(val_months[0].date()),
    "val_to": str(val_months[-1].date()),
    "test_from": str(test_months[0].date()),
    "test_to": str(test_months[-1].date()),
    "n_train": len(train),
    "n_val": len(val),
    "n_test": len(test),
    "n_purged": len(purged),
    "feature_cols": feature_cols,
    "drop_cols": DROP_COLS_EFF,
    "impute": impute_stats,
}

for path, part in [(OUT_TRAIN, train), (OUT_VAL, val), (OUT_TEST, test)]:
    part.to_parquet(path, index=False)
    part.to_csv(path.with_suffix(".csv"), index=False)
    print("saved", path.name, "+", path.with_suffix(".csv").name, "rows", len(part))

with open(OUT_IMPUTE, "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2, default=str)
print("saved", OUT_IMPUTE.name)

split_summary

## 5. Gợi ý dùng khi train model

```python
import json
import pandas as pd

meta = json.loads(open("data/impute_stats_train.json", encoding="utf-8").read())
TARGET = meta["target"]
features = meta["feature_cols"]

train = pd.read_parquet("data/churn_train.parquet")
val = pd.read_parquet("data/churn_val.parquet")
test = pd.read_parquet("data/churn_test.parquet")

X_train, y_train = train[features], train[TARGET]
X_val, y_val = val[features], val[TARGET]
X_test, y_test = test[features], test[TARGET]
# metric: AUCPR + AUC
```